In [2]:
!pip install pandas scipy openpyxl matplotlib seaborn -q

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports successful.")

All imports successful.


In [4]:
from google.colab import files
import io

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Load the Annotations sheet
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

# Rename columns to short names for convenience
df = df.rename(columns={
    'trans_border':         'tb',
    'identity':             'identity',
    'cultural_continuity':  'cc',
    'narrative':            'narrative',
    'accuracy':             'accuracy',
})

# Score columns
SCORE_COLS = ['tb', 'identity', 'cc', 'narrative', 'accuracy']

# Compute total score (sum of 5 dimensions, max = 15)
df['total'] = df[SCORE_COLS].sum(axis=1)

print(f"Loaded {len(df)} responses")
print(f"Models:    {df['model'].unique().tolist()}")
print(f"Languages: {df['language'].unique().tolist()}")
print(f"Prompts:   {sorted(df['prompt_id'].unique().tolist())}")
print()
df[['prompt_id','model','language'] + SCORE_COLS + ['total']].head(8)

Saving frontier_v3_manual_annotations.csv to frontier_v3_manual_annotations.csv
Loaded 44 responses
Models:    ['GPT-5.1', 'DeepSeek-V3.2']
Languages: ['Chinese', 'English']
Prompts:   ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3']



,prompt_id,model,language,tb,identity,cc,narrative,accuracy,total
0,A1,GPT-5.1,Chinese,3,2,3,3,3,14
1,A1,GPT-5.1,English,3,2,3,2,3,13
2,A1,DeepSeek-V3.2,Chinese,1,1,1,1,2,6
3,A1,DeepSeek-V3.2,English,2,2,2,2,3,11
4,A2,GPT-5.1,Chinese,3,3,3,3,3,15
5,A2,GPT-5.1,English,3,3,3,3,3,15
6,A2,DeepSeek-V3.2,Chinese,2,2,2,2,2,10
7,A2,DeepSeek-V3.2,English,2,2,2,2,3,11


In [5]:
# ── Group means ───────────────────────────────────────────────────────────
group_means = (
    df.groupby(['model', 'language'])[SCORE_COLS + ['total']]
    .mean()
    .round(2)
)

# Reorder rows to match report table order
row_order = [
    ('GPT-5.1',       'Chinese'),
    ('GPT-5.1',       'English'),
    ('DeepSeek-V3.2', 'Chinese'),
    ('DeepSeek-V3.2', 'English'),
]
group_means = group_means.reindex(row_order)

# Rename columns for display
group_means.columns = ['Trans-border', 'Identity', 'Cultural Cont.', 'Narrative', 'Accuracy', 'Total /15']

print("Table 1 — Average Scores by Model and Language (max per dim = 3, total = 15)")
print("Scale: 1 = Poor  |  2 = Partial  |  3 = Good")
print()
print(group_means.to_string())

# ── Highlight: which dimension is lowest in each group? ───────────────────
print("\n── Weakest dimension per group ──")
dim_cols = ['Trans-border', 'Identity', 'Cultural Cont.', 'Narrative', 'Accuracy']
for idx in group_means.index:
    row = group_means.loc[idx, dim_cols]
    weakest = row.idxmin()
    print(f"  {idx[0]:20s} | {idx[1]:8s} → weakest: {weakest} ({row[weakest]:.2f})")

Table 1 — Average Scores by Model and Language (max per dim = 3, total = 15)
Scale: 1 = Poor  |  2 = Partial  |  3 = Good

                        Trans-border  Identity  Cultural Cont.  Narrative  Accuracy  Total /15
model         language                                                                        
GPT-5.1       Chinese           3.00      2.73            2.91       2.82      3.00      14.45
              English           2.82      2.73            2.82       2.73      3.00      14.09
DeepSeek-V3.2 Chinese           2.09      1.64            2.00       1.64      2.36       9.73
              English           2.45      1.91            2.27       2.09      2.82      11.55

── Weakest dimension per group ──
  GPT-5.1              | Chinese  → weakest: Identity (2.73)
  GPT-5.1              | English  → weakest: Identity (2.73)
  DeepSeek-V3.2        | Chinese  → weakest: Identity (1.64)
  DeepSeek-V3.2        | English  → weakest: Identity (1.91)


Mann-Whitney U Test (Model Origin Effect)

In [6]:
def mannwhitney_with_effect(group1_scores, group2_scores, label):
    """
    Run Mann-Whitney U test between two independent groups.
    Returns U statistic, two-sided p-value, and effect size r.
    """
    n1, n2 = len(group1_scores), len(group2_scores)
    U, p   = stats.mannwhitneyu(group1_scores, group2_scores, alternative='two-sided')
    # Effect size r: ranges from -1 to 1
    # Positive r means group1 tends to rank higher than group2
    r      = 1 - (2 * U) / (n1 * n2)

    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))

    print(f"\n── {label} ──")
    print(f"  n(GPT-5.1) = {n1}   |   n(DeepSeek-V3.2) = {n2}")
    print(f"  GPT mean   = {group1_scores.mean():.2f}")
    print(f"  DS  mean   = {group2_scores.mean():.2f}")
    print(f"  U = {U}   p = {p:.4f} {sig}   effect r = {r:.3f}")

    if abs(r) >= 0.5:
        size_label = "LARGE"
    elif abs(r) >= 0.3:
        size_label = "medium"
    else:
        size_label = "small"
    print(f"  Effect size interpretation: {size_label}")
    return U, p, r


print("=" * 60)
print("MANN-WHITNEY U TEST — Model Origin Effect")
print("H₀: GPT-5.1 and DeepSeek-V3.2 come from the same distribution")
print("H₁: The two models differ in representational quality")
print("=" * 60)

# ── Test 1: Chinese condition ─────────────────────────────────────────────
gpt_zh = df[(df['model'] == 'GPT-5.1')      & (df['language'] == 'Chinese')]['total'].values
ds_zh  = df[(df['model'] == 'DeepSeek-V3.2') & (df['language'] == 'Chinese')]['total'].values
U1, p1, r1 = mannwhitney_with_effect(gpt_zh, ds_zh, "Chinese condition (ZH queries)")

# ── Test 2: English condition ─────────────────────────────────────────────
gpt_en = df[(df['model'] == 'GPT-5.1')      & (df['language'] == 'English')]['total'].values
ds_en  = df[(df['model'] == 'DeepSeek-V3.2') & (df['language'] == 'English')]['total'].values
U2, p2, r2 = mannwhitney_with_effect(gpt_en, ds_en, "English condition (EN queries)")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("""
Both conditions show statistically significant differences with
large effect sizes (|r| > 0.5). This means:

  → Model origin is a real and large determinant of representational
    quality, regardless of which language was used to query.

  → The bias is NOT query-triggered: even when DeepSeek is queried
    in English, it still scores significantly lower than GPT-5.1.
    This points toward model-embedded rather than surface-level bias.
""")

MANN-WHITNEY U TEST — Model Origin Effect
H₀: GPT-5.1 and DeepSeek-V3.2 come from the same distribution
H₁: The two models differ in representational quality

── Chinese condition (ZH queries) ──
  n(GPT-5.1) = 11   |   n(DeepSeek-V3.2) = 11
  GPT mean   = 14.45
  DS  mean   = 9.73
  U = 109.5   p = 0.0009 ***   effect r = -0.810
  Effect size interpretation: LARGE

── English condition (EN queries) ──
  n(GPT-5.1) = 11   |   n(DeepSeek-V3.2) = 11
  GPT mean   = 14.09
  DS  mean   = 11.55
  U = 99.5   p = 0.0086 **   effect r = -0.645
  Effect size interpretation: LARGE

INTERPRETATION

Both conditions show statistically significant differences with
large effect sizes (|r| > 0.5). This means:

  → Model origin is a real and large determinant of representational
    quality, regardless of which language was used to query.

  → The bias is NOT query-triggered: even when DeepSeek is queried
    in English, it still scores significantly lower than GPT-5.1.
    This points toward model-embe

Wilcoxon Signed-Rank Test (Language Effect)

In [7]:
def wilcoxon_language_effect(model_name):
    """
    Test whether Chinese vs English responses differ significantly
    within a single model, using Wilcoxon signed-rank test on
    prompt-matched pairs.
    """
    zh_scores = (
        df[(df['model'] == model_name) & (df['language'] == 'Chinese')]
        .sort_values('prompt_id')['total'].values
    )
    en_scores = (
        df[(df['model'] == model_name) & (df['language'] == 'English')]
        .sort_values('prompt_id')['total'].values
    )

    n_pairs = len(zh_scores)
    diffs   = zh_scores - en_scores

    # Wilcoxon requires at least one non-zero difference
    if np.all(diffs == 0):
        print(f"\n── {model_name} ──")
        print("  All ZH–EN differences are zero. Cannot run Wilcoxon.")
        return None, None

    W, p = stats.wilcoxon(zh_scores, en_scores)
    sig  = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))

    print(f"\n── {model_name} ──")
    print(f"  n pairs = {n_pairs}  (one pair per prompt)")
    print(f"  ZH scores: {zh_scores}")
    print(f"  EN scores: {en_scores}")
    print(f"  ZH − EN diffs: {diffs}")
    print(f"  ZH mean = {zh_scores.mean():.2f}  |  EN mean = {en_scores.mean():.2f}")
    print(f"  W = {W}   p = {p:.4f} {sig}")
    return W, p


print("=" * 60)
print("WILCOXON SIGNED-RANK TEST — Query Language Effect")
print("H₀: Chinese and English responses have the same score distribution")
print("H₁: Query language systematically shifts scores within each model")
print("=" * 60)

W1, p_gpt = wilcoxon_language_effect('GPT-5.1')
W2, p_ds  = wilcoxon_language_effect('DeepSeek-V3.2')

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("""
Neither model shows a significant language effect (both p > 0.05).

GPT-5.1:       p = 0.25  → not significant
DeepSeek-V3.2: p = 0.12  → not significant

Caveat for GPT-5.1: The p-value may reflect insufficient score variance
(ceiling effect) rather than a true absence of language effects.
73% of GPT responses are already at maximum score, leaving little
room to detect subtle language-driven differences on a 1–3 scale.

Combined with the Mann-Whitney results, the picture is clear:
  → Model origin effect: SIGNIFICANT (large)
  → Language effect within each model: NOT significant
  → Conclusion: bias is model-embedded, not query-triggered (RQ2 answer)
""")

WILCOXON SIGNED-RANK TEST — Query Language Effect
H₀: Chinese and English responses have the same score distribution
H₁: Query language systematically shifts scores within each model

── GPT-5.1 ──
  n pairs = 11  (one pair per prompt)
  ZH scores: [14 15 12 15 15 15 15 15 15 13 15]
  EN scores: [13 15 11 15 15 15 15 15 15 11 15]
  ZH − EN diffs: [1 0 1 0 0 0 0 0 0 2 0]
  ZH mean = 14.45  |  EN mean = 14.09
  W = 0.0   p = 0.2500 ns

── DeepSeek-V3.2 ──
  n pairs = 11  (one pair per prompt)
  ZH scores: [ 6 10  8 12 14  6  8 13  8  7 15]
  EN scores: [11 11 12 14 15 10 12  6 14  9 13]
  ZH − EN diffs: [-5 -1 -4 -2 -1 -4 -4  7 -6 -2  2]
  ZH mean = 9.73  |  EN mean = 11.55
  W = 15.0   p = 0.1162 ns

INTERPRETATION

Neither model shows a significant language effect (both p > 0.05).

GPT-5.1:       p = 0.25  → not significant
DeepSeek-V3.2: p = 0.12  → not significant

Caveat for GPT-5.1: The p-value may reflect insufficient score variance
(ceiling effect) rather than a true absence of l

In [8]:
# Flag severe ossification: identity=1 AND narrative=1
df['severe_ossification'] = (df['identity'] == 1) & (df['narrative'] == 1)

ossification_summary = (
    df.groupby(['model', 'language'])['severe_ossification']
    .agg(count='sum', total='count', rate='mean')
    .round(3)
)
ossification_summary['rate_pct'] = (ossification_summary['rate'] * 100).round(1)

print("Severe Identity Ossification (Identity=1 AND Narrative=1 simultaneously)")
print()
print(ossification_summary[['count', 'total', 'rate_pct']].rename(
    columns={'count': 'Ossified', 'total': 'Total', 'rate_pct': 'Rate (%)'}
).to_string())

print("\nKey takeaway:")
print("  DeepSeek-V3.2 Chinese: 55% severe ossification")
print("  DeepSeek-V3.2 English: 18% severe ossification  ← ossification appears")
print("  GPT-5.1 (both languages): 0% severe ossification")
print()
print("  The fact that DeepSeek shows ossification even in English")
print("  confirms the bias is model-embedded, not language-triggered.")

# Which specific prompts triggered ossification in DeepSeek?
print("\nOssified responses — DeepSeek-V3.2:")
ossified = df[(df['model'] == 'DeepSeek-V3.2') & (df['severe_ossification'])]
print(ossified[['prompt_id', 'category', 'language', 'tb', 'identity', 'cc', 'narrative', 'accuracy', 'total']]
      .sort_values(['language', 'prompt_id'])
      .to_string(index=False))

Severe Identity Ossification (Identity=1 AND Narrative=1 simultaneously)

                        Ossified  Total  Rate (%)
model         language                           
DeepSeek-V3.2 Chinese          6     11      54.5
              English          2     11      18.2
GPT-5.1       Chinese          0     11       0.0
              English          0     11       0.0

Key takeaway:
  DeepSeek-V3.2 Chinese: 55% severe ossification
  DeepSeek-V3.2 English: 18% severe ossification  ← ossification appears
  GPT-5.1 (both languages): 0% severe ossification

  The fact that DeepSeek shows ossification even in English
  confirms the bias is model-embedded, not language-triggered.

Ossified responses — DeepSeek-V3.2:
prompt_id category language  tb  identity  cc  narrative  accuracy  total
       A1        A  Chinese   1         1   1          1         2      6
       A3        A  Chinese   2         1   1          1         3      8
       B3        B  Chinese   1         1   1          

In [9]:
gpt_df = df[df['model'] == 'GPT-5.1'].copy()
ds_df  = df[df['model'] == 'DeepSeek-V3.2'].copy()

print("=" * 55)
print("CEILING EFFECT DIAGNOSIS")
print("=" * 55)

# Total score ceiling
gpt_ceiling = (gpt_df['total'] == 15).sum()
print(f"\nGPT-5.1 responses at maximum total (15/15): {gpt_ceiling} / {len(gpt_df)}  "
      f"({100*gpt_ceiling/len(gpt_df):.0f}%)")

# Per-dimension
print("\nGPT-5.1 — proportion at maximum (score=3) per dimension:")
dim_labels = {'tb':'Trans-border', 'identity':'Identity',
              'cc':'Cultural Cont.', 'narrative':'Narrative', 'accuracy':'Accuracy'}
for col, label in dim_labels.items():
    n_max = (gpt_df[col] == 3).sum()
    print(f"  {label:18s}: {n_max:2d}/{len(gpt_df)}  ({100*n_max/len(gpt_df):.0f}%)")

# Score distribution comparison
print("\nTotal score distribution:")
print(f"  {'Score':>6} | {'GPT-5.1':>10} | {'DeepSeek-V3.2':>15}")
print("  " + "-" * 36)
for s in range(5, 16):
    g = (gpt_df['total'] == s).sum()
    d = (ds_df['total']  == s).sum()
    print(f"  {s:>6} | {g:>10} | {d:>15}")

print("""
Why this matters for cross-validation:

  The near-zero Pearson r in cross-validation (r = 0.015) is
  structurally explained by this ceiling effect:

  • Pairwise SCORE DIFFERENCES are dominated by model identity
    (GPT vs DeepSeek) because GPT scores are compressed at ceiling.

  • Pairwise EMBEDDING SIMILARITY is dominated by query language
    (same-language pairs cluster together regardless of model).

  These two variables are driven by DIFFERENT experimental factors,
  so they have little structural opportunity to co-vary → r ≈ 0.

  This is NOT evidence that embeddings cannot detect frame-level bias.
  It is evidence that this dataset's structure prevents a clean test.
  → Phase 2 agenda: purpose-built minimal pairs needed.
""")

CEILING EFFECT DIAGNOSIS

GPT-5.1 responses at maximum total (15/15): 16 / 22  (73%)

GPT-5.1 — proportion at maximum (score=3) per dimension:
  Trans-border      : 20/22  (91%)
  Identity          : 16/22  (73%)
  Cultural Cont.    : 19/22  (86%)
  Narrative         : 17/22  (77%)
  Accuracy          : 22/22  (100%)

Total score distribution:
   Score |    GPT-5.1 |   DeepSeek-V3.2
  ------------------------------------
       5 |          0 |               0
       6 |          0 |               3
       7 |          0 |               1
       8 |          0 |               3
       9 |          0 |               1
      10 |          0 |               2
      11 |          2 |               2
      12 |          1 |               3
      13 |          2 |               2
      14 |          1 |               3
      15 |         16 |               2

Why this matters for cross-validation:

  The near-zero Pearson r in cross-validation (r = 0.015) is
  structurally explained by this 

In [10]:
print("Per-Prompt Score Gap: GPT-5.1 minus DeepSeek-V3.2")
print("(Positive = GPT scored higher; max possible gap = 10)")
print()

for lang in ['Chinese', 'English']:
    gpt_l = (
        df[(df['model'] == 'GPT-5.1') & (df['language'] == lang)]
        .sort_values('prompt_id')
        [['prompt_id', 'category', 'total']]
        .rename(columns={'total': 'GPT'})
        .reset_index(drop=True)
    )
    ds_l = (
        df[(df['model'] == 'DeepSeek-V3.2') & (df['language'] == lang)]
        .sort_values('prompt_id')
        [['prompt_id', 'total']]
        .rename(columns={'total': 'DS'})
        .reset_index(drop=True)
    )
    merged = pd.concat([gpt_l, ds_l[['DS']]], axis=1)
    merged['Gap'] = merged['GPT'] - merged['DS']

    print(f"── {lang} ──")
    print(merged[['prompt_id', 'category', 'GPT', 'DS', 'Gap']]
          .sort_values('Gap', ascending=False)
          .to_string(index=False))
    print(f"  Average gap: {merged['Gap'].mean():.2f}")
    print(f"  Max gap at:  {merged.loc[merged['Gap'].idxmax(), 'prompt_id']} "
          f"(gap = {merged['Gap'].max()})")
    print()

Per-Prompt Score Gap: GPT-5.1 minus DeepSeek-V3.2
(Positive = GPT scored higher; max possible gap = 10)

── Chinese ──
prompt_id category  GPT  DS  Gap
       B3        B   15   6    9
       A1        A   14   6    8
       C1        C   15   8    7
       D1        D   15   8    7
       D2        D   13   7    6
       A2        A   15  10    5
       A3        A   12   8    4
       B1        B   15  12    3
       C2        C   15  13    2
       B2        B   15  14    1
       D3        D   15  15    0
  Average gap: 4.73
  Max gap at:  B3 (gap = 9)

── English ──
prompt_id category  GPT  DS  Gap
       C2        C   15   6    9
       B3        B   15  10    5
       A2        A   15  11    4
       C1        C   15  12    3
       A1        A   13  11    2
       D3        D   15  13    2
       D2        D   11   9    2
       D1        D   15  14    1
       B1        B   15  14    1
       B2        B   15  15    0
       A3        A   11  12   -1
  Average gap: 2.55
  Max 

In [11]:
print("=" * 65)
print("SUMMARY — Manual Coding Statistical Results")
print("=" * 65)

print("""
── Model Origin Effect (Mann-Whitney U, independent groups) ──

  Condition    │  U      │  p       │  sig  │  effect r  │  interpretation
  ─────────────┼─────────┼──────────┼───────┼────────────┼─────────────────
  Chinese      │  109.5  │  0.001   │  ***  │  0.705     │  LARGE
  English      │   99.5  │  0.009   │  **   │  0.560     │  LARGE

  → Origin effect is significant and large in BOTH languages.

── Query Language Effect (Wilcoxon signed-rank, paired) ──────

  Model           │  W     │  p       │  sig  │  interpretation
  ────────────────┼────────┼──────────┼───────┼─────────────────
  GPT-5.1         │  0.0   │  0.250   │  ns   │  not significant
  DeepSeek-V3.2   │  15.0  │  0.116   │  ns   │  not significant

  → Language effect is NOT significant within either model.
  Note: GPT result may reflect ceiling effect rather than true
        absence of language influence.

── Answer to RQ2 ─────────────────────────────────────────────

  Identity ossification is MODEL-EMBEDDED, not query-triggered.
  The nation-state interpretive framework is carried by the model
  regardless of query language.

  Effect size benchmarks (Cohen): small ≥ 0.1 | medium ≥ 0.3 | large ≥ 0.5
  Significance: *** p<0.001  ** p<0.01  * p<0.05  ns p≥0.05
""")

SUMMARY — Manual Coding Statistical Results

── Model Origin Effect (Mann-Whitney U, independent groups) ──

  Condition    │  U      │  p       │  sig  │  effect r  │  interpretation
  ─────────────┼─────────┼──────────┼───────┼────────────┼─────────────────
  Chinese      │  109.5  │  0.001   │  ***  │  0.705     │  LARGE
  English      │   99.5  │  0.009   │  **   │  0.560     │  LARGE

  → Origin effect is significant and large in BOTH languages.

── Query Language Effect (Wilcoxon signed-rank, paired) ──────

  Model           │  W     │  p       │  sig  │  interpretation
  ────────────────┼────────┼──────────┼───────┼─────────────────
  GPT-5.1         │  0.0   │  0.250   │  ns   │  not significant
  DeepSeek-V3.2   │  15.0  │  0.116   │  ns   │  not significant

  → Language effect is NOT significant within either model.
  Note: GPT result may reflect ceiling effect rather than true
        absence of language influence.

── Answer to RQ2 ────────────────────────────────────────